Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from google.colab import files

Loading Dataset

In [ ]:
df = pd.read_csv('Linear _1 assement.csv')

Data Cleaning

In [ ]:
df = pd.read_csv('Linear _1 assement.csv')
df = df.drop(columns=['fish_id'])
df['species'] = df['species'].astype(str).str.strip().str.capitalize()

# Log transform the target to prevent negative predictions
# We predict log(weight) then convert back using exp()
df = df[df['weight_g'] > 0] # Ensure no zero weights
df['log_weight'] = np.log(df['weight_g'])

OUTLIER REMOVAL

In [ ]:
df = df[df['weight_g'] < df['weight_g'].quantile(0.95)]

Feature Engineering

In [ ]:
df_model = df.drop(columns=['length1_cm', 'length2_cm', 'weight_g'])
df_encoded = pd.get_dummies(df_model, columns=['species'], drop_first=True)

X/y Split

In [ ]:
X = df_encoded.drop(columns=['log_weight'])
y = df_encoded['log_weight']

Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Professional Scaling

In [ ]:
X_mean, X_std = X_train.mean(), X_train.std()
X_train_scaled = (X_train - X_mean) / X_std
X_test_scaled = (X_test - X_mean) / X_std

Model Building

In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

Evaluation

In [ ]:
y_pred_log = model.predict(X_test_scaled)
y_pred_actual = np.exp(y_pred_log)
y_test_actual = np.exp(y_test)
print(f"✅ Final R² Score: {r2_score(y_test_actual, y_pred_actual):.4f}")
print(f"✅ Final MAE: {mean_absolute_error(y_test_actual, y_pred_actual):.2f}g")

✅ Final R² Score: 0.7621
✅ Final MAE: 996.72g


Export

In [ ]:
assets = {
    'model': model,
    'columns': X.columns.tolist(),
    'mean': X_mean,
    'std': X_std,
    'species_list': sorted(df['species'].unique().tolist()),
    'use_log': True # Flag for the app to know to use np.exp()
}

with open('fish_model_elite.pkl', 'wb') as f:
    pickle.dump(assets, f)

files.download('fish_model_elite.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>